# Lecture 10 · Notebook 2 — Chat with an open multimodal LLM
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture10/lecture10_2_chat_with_an_mllm.ipynb)

**Companion to [Parts 3–4 of the video lecture](https://nlp-course-cuhksz.github.io/lecture10.html#p3-s46)** (architecture and training). Modern multimodal LLMs follow the
recipe from the lecture: a **vision encoder** turns the image into patch features, a **connector** maps them into the LLM's embedding
space, and the **LLM** reads image tokens and text together. Here you will:

1. ask an open model (Qwen2.5-VL-3B-Instruct) questions about images,
2. count how many **visual tokens** an image becomes, and
3. look inside the model to find the encoder, connector and LLM.

**Use a GPU:** *Runtime → Change runtime type → T4 GPU*. The 3B model needs about 8 GB of GPU memory in half precision.
On a CPU, switch `MODEL_ID` to the tiny `HuggingFaceTB/SmolVLM-256M-Instruct`. If the 3B model gives errors or garbled text on your
GPU, try `HuggingFaceTB/SmolVLM2-2.2B-Instruct`.

In [ ]:
!pip -q install -U transformers accelerate

In [ ]:
import torch, requests, io
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"       # on CPU: "HuggingFaceTB/SmolVLM-256M-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device).eval()
print(MODEL_ID, "on", device)

def load(url):
    return Image.open(io.BytesIO(requests.get(url, timeout=30).content)).convert("RGB")

cats = load("http://images.cocodataset.org/val2017/000000039769.jpg")
street = load("http://images.cocodataset.org/val2017/000000000724.jpg")
cats

A small helper: build a chat message with images and text, apply the model's chat template, generate, and decode only the new tokens.

In [ ]:
@torch.no_grad()
def ask(images, question, max_new_tokens=128):
    images = images if isinstance(images, list) else [images]
    messages = [{"role": "user", "content": [{"type": "image", "image": im} for im in images] + [{"type": "text", "text": question}]}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

print(ask(cats, "Describe this image in two sentences."))

## 1 · Visual question answering — try your own questions

In [ ]:
for q in ["How many cats are there?", "What objects are lying next to the cats?", "What colour is the blanket?"]:
    print("Q:", q, "\nA:", ask(cats, q), "\n")
print(ask(street, "What does the sign say, and what should a driver do?"))

## 2 · Several images in one conversation

In [ ]:
print(ask([cats, street], "Compare the two images: what is the main object in each, and which one shows an outdoor scene?"))

## 3 · How many tokens is an image?
The vision encoder cuts the image into patches; the connector turns them into tokens the LLM reads like words. Count them by
comparing the prompt length with and without the image. Qwen2.5-VL keeps the native resolution, so a smaller image becomes fewer
tokens; some small models (SmolVLM) resize every image to a fixed grid, so their count does not change.

In [ ]:
def n_tokens(content):
    msgs = [{"role": "user", "content": content}]
    x = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt")
    return x["input_ids"].shape[1]

text_only = n_tokens([{"type": "text", "text": "Describe the image."}])
with_image = n_tokens([{"type": "image", "image": cats}, {"type": "text", "text": "Describe the image."}])
print(f"image size {cats.size}: about {with_image - text_only} visual tokens")
small = cats.resize((cats.width // 2, cats.height // 2))
with_small = n_tokens([{"type": "image", "image": small}, {"type": "text", "text": "Describe the image."}])
print(f"image size {small.size}: about {with_small - text_only} visual tokens")

## 4 · Inside the model: encoder, connector, LLM
Group the parameters by their top-level module names. You should find a vision tower, a projection (connector) and the language model —
the three boxes from Part 3 of the lecture.

In [ ]:
from collections import Counter
sizes = Counter()
for name, p in model.named_parameters():
    parts = name.split(".")
    key = ".".join(parts[:2]) if parts[0] == "model" else parts[0]
    sizes[key] += p.numel()
for k, v in sizes.most_common():
    print(f"{k:40s} {v / 1e6:8.1f} M parameters")
# the connector may live inside the vision tower (Qwen calls it "merger")
conn = sum(p.numel() for n, p in model.named_parameters() if any(w in n for w in ("merger", "connector", "projector", "multi_modal_projector")))
print(f"{'connector (merger / projector)':40s} {conn / 1e6:8.1f} M parameters")

## Exercises
1. Upload one of your own photos (*Files* panel on the left) and ask the model about it. Where does it fail?
2. Ask for a JSON list of the objects with a rough location (left / middle / right). Is the output valid JSON?
3. Halve the image again in section 3. How do the token count and the answer quality change? This is the *resolution vs. cost* trade-off.
4. Which group in section 4 is the largest? Relate this to the lecture: why is the LLM usually the biggest part?

**Next:** [Notebook 3 — speech as tokens](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture10/lecture10_3_speech_as_tokens.ipynb)